In [1]:
"""
We will be creating a conversational chat bot. 
We will be using a website and we will be asking questions to our model,
based on the context of that website.
"""
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] =os.getenv("HF_TOKEN")

llm = ChatGroq(model = "llama-3.1-8b-instant", groq_api_key=groq_api_key)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000021973F9BFD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021973FCC070>, model_name='llama-3.1-8b-instant', groq_api_key=SecretStr('**********'))

In [2]:
import langchain
print(langchain.__version__)

0.1.17


In [23]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains.retrieval import create_retrieval_chain
#from langchain.chains.combine_documents import create_stuff_documents_chain

In [5]:
embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")

c:\Users\KIIT\anaconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2550.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
#Load, Chunk and index the contents of the page to create a retriever
import bs4
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

#Every LLM model has chunk_size limitation, so it is better to divide it into different chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
documents_chunk = text_splitter.split_documents(docs)

from langchain_core.documents import Document

import uuid

texts = [doc.page_content for doc in documents_chunk]
metadatas = [doc.metadata for doc in documents_chunk]
ids = [str(uuid.uuid4()) for _ in documents_chunk]

db = Chroma.from_texts(
    texts=texts,
    metadatas=metadatas,
    ids=ids,
    embedding=embeddings
)

#db = Chroma.from_documents(documents_chunk, embeddings)
retriever = db.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002191484CB80>)

In [14]:
#Prompt Template
system_prompt = (
    "You are an assistant for question-answering tasks."
    "Use the following pieces of retrieved context to answer"
    "the question. If you don't know the answer, say that you"
    "don't know. Use 3 sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [49]:
"""#question_answer_chain = create_stuff_document_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
response = rag_chain.invoke(
    {"input": "What is self reflection"}
)
response.content"""
#we cannot use the above 2 (create_suff_document_chain and create_retrieval_chain) as I am using Langchain version 1.x.y.
#The above only works for Langchain version <= 0.x.y. Thats why we will use the below method (Manual)
#pip install langchain==0.1.17

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def rag_chain(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)

    formatted_prompt = prompt.invoke({
        "context": context,
        "input": question
    })

    response = llm.invoke(formatted_prompt)
    return response

In [29]:
response = rag_chain("What is self reflection?")
print(response.content)

Self-reflection is a vital aspect that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It plays a crucial role in real-world tasks where trial and error are inevitable. It involves analyzing past actions and decisions to guide future changes in the plan.


In [30]:
response = rag_chain("Explain agent system overview")
response.content

'The agent system overview includes two key components: Planning and Memory. \n\nPlanning involves the agent breaking down large tasks into smaller, manageable subgoals, and then reflecting and refining past actions to improve the quality of final results. \n\nMemory, on the other hand, relies on a natural language interface between the Large Language Model (LLM) and external components, but this interface has reliability issues, such as formatting errors and rebellious behavior.'

In [ ]:
#Adding a chat history
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do not answer the question, "
    "just reformulate it if needed and otherwise return it as it is."
)

chat_history = []

def contextualize_question(question, chat_history):
    messages = [
        ("system", contextualize_q_system_prompt),
        ("human", f"Chat history:\n{chat_history}\n\nQuestion: {question}")
    ]

    prompt = ChatPromptTemplate.from_messages(messages)
    formatted = prompt.invoke({})

    response = llm.invoke(formatted)
    return response.content

def retrieve_with_history(question, chat_history):
    standalone_question = contextualize_question(question, chat_history)
    docs = retriever.invoke(standalone_question)
    return docs, standalone_question

def rag_with_memory(question):
    global chat_history

    docs, refined_question = retrieve_with_history(question, chat_history)

    context = "\n\n".join(doc.page_content for doc in docs)

    formatted_prompt = prompt.invoke({
        "context": context,
        "input": question
    })

    response = llm.invoke(formatted_prompt)

    # update memory
    chat_history.append(f"User: {question}")
    chat_history.append(f"Assistant: {response.content}")

    return response

response = rag_with_memory("What is self reflection?")
print(response.content)


Self-reflection is a vital aspect that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It helps autonomous systems learn from their experiences and make better decisions in the future.


In [45]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "Question: {input}\n\nContext:\n{context}")
    ]
)

def question_answer_chain(input, chat_history, docs):
    # combine retrieved docs
    context = "\n\n".join(doc.page_content for doc in docs)

    # build prompt
    formatted_prompt = qa_prompt.invoke({
        "input": input,
        "chat_history": chat_history,
        "context": context
    })

    # call LLM
    response = llm.invoke(formatted_prompt)
    return response

def rag_chain_manual(question, chat_history):
    # retrieve docs using chat-aware retriever
    docs = retriever.invoke(question)

    # get answer
    response = question_answer_chain(question, chat_history, docs)

    return response

chat_history = []

question1 = "what is self reflection?"
response1 = rag_chain_manual(question1, chat_history)

chat_history.extend([
    HumanMessage(content=question1),
    AIMessage(content=response1.content)
])

question2 = "Tell me more about it"

response2 = rag_chain_manual(question2, chat_history)

print(response2.content)

Self-reflection in the context of autonomous agents involves analyzing past actions and decisions to identify areas for improvement. It uses a Large Language Model (LLM) to evaluate failed trajectories and provide an ideal reflection for guiding future changes in the plan. This reflection is then stored in the agent's working memory to inform future decisions, allowing the agent to adapt and learn from its mistakes.


In [48]:
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = []
    return store[session_id]

def rag_with_memory(question, session_id):
    # get memory
    history = get_session_history(session_id)

    # retrieve documents
    docs = retriever.invoke(question)

    # build context
    context = "\n\n".join(doc.page_content for doc in docs)

    # format prompt
    formatted_prompt = qa_prompt.invoke({
        "input": question,
        "chat_history": history,
        "context": context
    })

    # get answer from LLM
    response = llm.invoke(formatted_prompt)

    # update memory
    history.append(("human", question))
    history.append(("ai", response.content))

    return response.content

print(rag_with_memory("What is task decomposition?", "abc123"))
print(rag_with_memory("What are common ways of doing it?", "abc123"))

Task decomposition is the process of breaking down a complex task into smaller, more manageable tasks or subgoals. It involves identifying the steps needed to achieve a goal and planning ahead to ensure successful completion. This process can be done through various methods, including using LLMs with simple prompting, task-specific instructions, or human inputs.
Based on the provided context, there are two common ways of task decomposition:

1. **Chain of Thought (CoT)**: This method involves instructing the model to "think step by step" to decompose hard tasks into smaller and simpler steps. It transforms big tasks into multiple manageable tasks and sheds light into the model's thinking process.

2. **Tree of Thoughts**: This method extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps, generates multiple thoughts per step, and creates a tree structure. The search process can be BFS or DFS, with each state e